# DocVQA QLoRA — Colab runner
This notebook only **sets up the environment and calls the scripts** in `src/`. All logic lives in the repo.

Runtime → Change runtime type → **T4 GPU**. Run the cells top to bottom. Each phase has a **smoke test** cell (`--limit`) — run it first, then the full cell.

If the runtime disconnects: re-run cells **1–5** (setup), then continue where you stopped. Training resumes automatically from the last checkpoint on Drive.

## 1. Check GPU

In [ ]:
!nvidia-smi
import torch
print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
assert torch.cuda.is_available(), "No GPU! Runtime > Change runtime type > T4 GPU"
major, minor = torch.cuda.get_device_capability(0)
print(torch.cuda.get_device_name(0), f"capability {major}.{minor}", "-> bf16" if major >= 8 else "-> fp16 (no native bf16)")

## 2. Mount Google Drive (all outputs are stored here)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Paths + HF cache on Drive
Edit `REPO_URL` to your GitHub repo.

In [ ]:
import os
REPO_URL = "https://github.com/YOUR_GITHUB_USERNAME/docvqa-qlora.git"   # <-- EDIT
REPO     = "/content/docvqa-qlora"
DRIVE    = "/content/drive/MyDrive/docvqa-qlora"
DATA_DRIVE = f"{DRIVE}/data/processed"       # dataset (persistent)
DATA     = "/content/data/processed"         # fast local copy used for training/eval
CKPT     = f"{DRIVE}/outputs/qlora"          # checkpoints + best_adapter (persistent)
RESULTS  = f"{DRIVE}/results"                # every reported number (persistent)
MERGED   = "/content/models/merged"          # ~7.5 GB: local disk, NOT Drive (15 GB free quota)
os.environ["HF_HOME"] = f"{DRIVE}/hf_cache"  # model download (~7.5 GB) survives restarts
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "0"
for p in (DRIVE, RESULTS, CKPT, os.environ["HF_HOME"]):
    os.makedirs(p, exist_ok=True)
print("HF_HOME =", os.environ["HF_HOME"])

## 4. Clone (or update) the repo

In [ ]:
if not os.path.exists(REPO):
    !git clone {REPO_URL} {REPO}
else:
    !git -C {REPO} pull
%cd {REPO}
!git log --oneline -1

## 5. Install pinned requirements (torch stays the Colab build)
If pip prints *"you must restart the runtime"*: Runtime → Restart session, then re-run cells 3–4 (skip 5).

In [ ]:
!pip install -q -r requirements.txt
!python -c "import transformers, peft, bitsandbytes, datasets; print('transformers', transformers.__version__, '| peft', peft.__version__, '| bnb', bitsandbytes.__version__, '| datasets', datasets.__version__)"

## 6. Unit tests (CPU-level checks, ~1 min)

In [ ]:
!python -m pytest -q

---
## Phase 1 — Data
First print the distinct `question_types`, then a 50-sample smoke test, then the full ~3000 samples (≈10–20 min).

In [ ]:
!python -m src.data.prepare_dataset --config configs/data.yaml --list_types 2000

In [ ]:
!python -m src.data.prepare_dataset --config configs/data.yaml --limit 50 --output_dir /content/data_smoke

In [ ]:
!python -m src.data.prepare_dataset --config configs/data.yaml --output_dir {DATA_DRIVE}

Copy the dataset from Drive to local disk (reading thousands of images from Drive is slow). **Re-run this after every runtime restart.**

In [ ]:
!mkdir -p /content/data && rsync -a {DATA_DRIVE}/ {DATA}/
!wc -l {DATA}/*.jsonl && du -sh {DATA}

---
## Phase 3 — Baseline evaluation (base model, fp16 on T4)
Smoke test on 20 samples first (~2 min incl. model download), then the full test split (~10–20 min).

In [ ]:
!python -m src.evaluate --config configs/eval.yaml --split test --data_dir {DATA} --limit 20 --output {RESULTS}/smoke_baseline.json

In [ ]:
!python -m src.evaluate --config configs/eval.yaml --split test --data_dir {DATA} --output {RESULTS}/baseline.json

Optional, for an apples-to-apples comparison with the 4-bit fine-tuned model:

In [ ]:
# !python -m src.evaluate --config configs/eval.yaml --split test --data_dir {DATA} --load_in_4bit --output {RESULTS}/baseline_4bit.json

---
## Phase 4 — QLoRA training
Smoke test: 32 samples, 1 epoch (2 optimizer steps) — checks no OOM, loss is finite, callback + checkpointing work.

In [ ]:
!python -m src.train --config configs/train.yaml --data_dir {DATA} --limit 32 --val_limit 8 \
    --output_dir /content/smoke_ckpt --results_dir /content/smoke_results --no_resume \
    --set training.num_train_epochs=1 training.save_steps=1000

Full run (~2400 samples × 2 epochs; a few hours on a T4). **If Colab disconnects, re-run setup cells 1–5 + the rsync cell, then this cell again — it resumes from the last checkpoint on Drive.**

In [ ]:
!python -m src.train --config configs/train.yaml --data_dir {DATA} --output_dir {CKPT} --results_dir {RESULTS}

In [ ]:
!cat {RESULTS}/val_anls_per_epoch.json
!cat {RESULTS}/trainable_params.json
!ls {CKPT}

---
## Phase 5 — Final evaluation on TEST + comparison
Same prompt / pixels / decoding as the baseline; the adapter is loaded on the 4-bit base exactly as trained.

In [ ]:
!python -m src.evaluate --config configs/eval.yaml --split test --data_dir {DATA} --adapter {CKPT}/best_adapter --load_in_4bit --output {RESULTS}/finetuned.json

In [ ]:
!python -m src.compare --baseline {RESULTS}/baseline.json --finetuned {RESULTS}/finetuned.json --results_dir {RESULTS}

---
## Phase 6 — Merge the adapter (fp16 base + LoRA → one model)
Verifies merged vs unmerged ANLS on 100 val samples.

In [ ]:
!python -m src.merge_adapter --config configs/eval.yaml --adapter {CKPT}/best_adapter --output {MERGED} \
    --data_dir {DATA} --results_dir {RESULTS} --verify_limit 100
!cat {RESULTS}/merge_check.json

Optional: test-set score of the merged fp16 model (same precision as the baseline).

In [ ]:
# !python -m src.evaluate --config configs/eval.yaml --split test --data_dir {DATA} --model_path {MERGED} --output {RESULTS}/finetuned_merged.json

---
## Phase 7 — Run the REST API inside Colab and call it

In [ ]:
import subprocess, time, requests
env = dict(os.environ, MODEL_MODE="merged", MERGED_MODEL_PATH=MERGED)
server = subprocess.Popen(["uvicorn", "src.serve.app:app", "--host", "0.0.0.0", "--port", "8000"],
                          env=env, stdout=open("/content/server.log", "w"), stderr=subprocess.STDOUT)
for _ in range(120):
    try:
        if requests.get("http://localhost:8000/health", timeout=2).json().get("model_loaded"):
            break
    except Exception:
        pass
    time.sleep(5)
print(requests.get("http://localhost:8000/health").json())

In [ ]:
import json
row = json.loads(open(f"{DATA}/test.jsonl").readline())
img = f"{DATA}/{row['image']}"
print("Q:", row["question"], "| gold:", row["answers"])
!curl -s -X POST http://localhost:8000/predict -F "file=@{img}" -F "question={row['question']}"

In [ ]:
from IPython.display import Image as IPImage, display
display(IPImage(img, width=500))
r = requests.post("http://localhost:8000/predict", files={"file": open(img, "rb")}, data={"question": row["question"]})
print(r.status_code, r.json())
print(requests.post("http://localhost:8000/predict", files={"file": ("a.txt", b"hello")}, data={"question": "x"}).json())  # 415 demo

In [ ]:
server.terminate(); !tail -n 20 /content/server.log

---
## Save artifacts
Download the results + adapter to commit `results/` to GitHub and to serve on another machine. (The merged model is 7.5 GB — re-create it with `src.merge_adapter` where you serve.)

In [ ]:
!cd {DRIVE} && zip -qr /content/docvqa_artifacts.zip results outputs/qlora/best_adapter -x "results/smoke*"
!ls -lh /content/docvqa_artifacts.zip
from google.colab import files
files.download("/content/docvqa_artifacts.zip")